# Regime Exploration — NZWNExplore the regime taxonomy using METAR wind / temperature / dewpoint.Compute ΔT/h, wind-direction change, dewpoint depression, and classify daysinto calm / transition / late_warming / foehn_nw / disrupted.Purely exploratory — not load-bearing.

In [ ]:
import polars as plimport datetime as dtfrom pathlib import Pathfrom solarstorm.data._iem import fetch_iem_asosfrom solarstorm.eda._regimes import classify_regimecache = Path('./.cache/iem')df = fetch_iem_asos('NZWN', dt.date(2009,1,1), dt.date(2026,6,3), cache_dir=cache)print(f'{df.height:,} rows')

In [ ]:
# Classify each local day into a regime, build a distribution table.df_local = df.with_columns(    pl.col('valid').dt.convert_time_zone('Pacific/Auckland').alias('ts_local'),)df_local = df_local.with_columns(pl.col('ts_local').dt.date().alias('date_local'))regimes = []for date_local, day_df in df_local.group_by('date_local'):    try:        regime, flags = classify_regime(day_df)    except Exception:        regime = 'unknown'    regimes.append({'date_local': date_local[0] if isinstance(date_local, tuple) else date_local, 'regime': regime})reg_df = pl.DataFrame(regimes)reg_df.group_by('regime').count().sort('count', descending=True)

In [ ]:
# Regime share by monthreg_df = reg_df.with_columns(pl.col('date_local').dt.month().alias('month'))reg_df.group_by(['month','regime']).count().sort(['month','count'], descending=[False, True])